<h1>🎛️ Biofilter — Report: <code>expand_variant_regulatory</code></h1>

Which genes a variant **regulates**, in which tissue, with what effect.

Not the same question as `annotate_variant`, which reports the gene a
variant sits *in*. They are usually different genes — see section 3.

### 1. Open a bundle

In [ ]:
from pathlib import Path

from biofilter import Biofilter

BUNDLE = None
REPORT = "expand_variant_regulatory"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)

_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bf

### 2. Which tissues does this bundle carry

Ask first. GTEx ships 50 tissues and the build loads a chosen subset, so
absence of evidence here means absence **in these tissues**.

In [ ]:
import duckdb

from biofilter.modules.report import Bundle

with Bundle.open(bf.core.db_uri.removeprefix("parquet://")) as bundle:
    tissues = bundle.con.execute(
        "SELECT bio_context, count(*) AS n FROM variant_gtex "
        "GROUP BY 1 ORDER BY n DESC"
    ).to_arrow_table().to_pandas()

print(f"{len(tissues)} tissues in this bundle")
tissues

### 3. The gene it is in, and the gene it regulates

This is the whole reason the report exists.

In [ ]:
result = bf.report.run(REPORT, input_data=["22:42914646"], p_value_max=1e-10)
df = result.to_pandas()

df[["variant_key", "position_gene_symbol", "regulated_gene_symbol",
    "bio_context", "beta", "p_value"]].head(8)

On chr22 of this bundle, of 6,919,646 variant × gene pairs carrying both
kinds of evidence, **88.5% name a different gene**. And 31,080 variants
with regulatory evidence are classed by VEP as `intergenic`, `upstream`
or `downstream` — outside any gene at all. For those, `annotate_variant`
says "not in a gene" while the eQTL says "regulates this one".

### 4. Three input shapes, mixed freely

| shape | means |
| --- | --- |
| `APOE` | every variant inside that gene's range |
| `rs429358` | that variant |
| `22:42914646` | that position |

Anything that is not an rsID or a position is read as a gene name.

In [ ]:
mixed = bf.report.run(
    REPORT,
    input_data=["DDT", "22:42914646", "rs4822455"],
    p_value_max=1e-20,
).to_pandas()

mixed.groupby(["input_value", "input_kind"]).size().to_frame("rows")

### 5. Gene mode: what do variants in this gene regulate

In [ ]:
gene = bf.report.run(REPORT, input_data=["DDT"], p_value_max=1e-50).to_pandas()

print(f"{len(gene):,} rows, "
      f"{gene['regulated_gene_id'].nunique()} regulated genes, "
      f"{gene['bio_context'].nunique()} tissues")

gene.groupby("regulated_gene_symbol", dropna=False).agg(
    tissues=("bio_context", "nunique"),
    best_p=("p_value", "min"),
).sort_values("best_p").head(10)

`flanking_bp` widens a gene's range, for promoter and downstream
regions.

In [ ]:
for flank in (0, 5000, 50000):
    out = bf.report.run(REPORT, input_data=["DDT"], flanking_bp=flank,
                        p_value_max=1e-50).to_pandas()
    print(f"  flanking_bp={flank:>6,}  {len(out):>6,} rows")

### 6. Narrowing by tissue and significance

Tissue names come from the data. Pass whatever the bundle has.

In [ ]:
one_tissue = bf.report.run(
    REPORT, input_data=["DDT"],
    tissues=[tissues.iloc[0]["bio_context"]],
    p_value_max=1e-50,
).to_pandas()

print(f"{tissues.iloc[0]['bio_context']}: {len(one_tissue):,} rows")
one_tissue[["regulated_gene_symbol", "beta", "se", "p_value", "n"]].head(6)

### 7. Two kinds of null, both meaningful

**`regulated_gene_symbol` null** — GTEx names its target by Ensembl id,
and roughly 17% of those on chr22 have no BF4 entity (lncRNAs and
pseudogenes without HGNC symbols). The evidence is real;
`regulated_gene_id` is always there.

**`position_gene_symbol` null** — the variant falls outside every gene
body. Common, expected, and exactly the case this report is for.

In [ ]:
wide = bf.report.run(REPORT, input_data=["22:23913109"]).to_pandas()

print("rows:", len(wide))
print("regulated genes without a BF4 symbol:",
      int(wide["regulated_gene_symbol"].isna().sum()))
print("variant outside any gene body:",
      bool(wide["position_gene_symbol"].isna().all()))

### 8. Export

In [ ]:
for path in result.write(OUTPUT_DIR / "expand_variant_regulatory.csv"):
    print(path)

### 9. The same thing on the command line

```bash
biofilter report run --report-name expand_variant_regulatory \\
    --input APOE \\
    --param p_value_max=1e-8 --param flanking_bp=5000 \\
    --output regulatory.csv
```